# Random entangled states via the Choi–Jamiołkowski isomorphism

Under the Choi–Jamiołkowski isomorphism a linear map Φ : M_{d_in} → M_{d_out}
corresponds to the operator (Φ ⊗ I)(|Φ⁺⟩⟨Φ⁺|) on C^{d_in} ⊗ C^{d_out}, where
|Φ⁺⟩ is the maximally entangled vector. Sampling a Haar-random unitary U on the
joint space and applying it to |Φ⁺⟩ gives a random pure state whose density
matrix we read as a Choi matrix.

A generic pure state has full Schmidt rank, so it is **NPT-entangled**: the
partial transpose has a negative eigenvalue. Unlike the UPB and
antisymmetric-subspace SDP generators (which produce *PPT* bound entangled
states), these are not inputs to the PPT² test. This notebook is a self-contained
demonstration of the Choi construction and the partial-transpose entanglement
criterion that the rest of the pipeline relies on.

In [ ]:
include("common.jl")

## Random Choi matrices

Apply a Haar-random unitary (`random_unitary` from `common.jl`) to the maximally
entangled vector and rescale so that the trace equals `dim_in`, the Choi
normalisation of a trace-preserving map.

In [ ]:
function random_entangled_choi(dim_in=2, dim_out=2; rng=Random.GLOBAL_RNG)
    d = dim_in * dim_out
    U = random_unitary(d; rng=rng)

    ϕ = zeros(ComplexF64, d)              # maximally entangled (1/√dim_in) Σ |i⟩⊗|i⟩
    for i in 1:min(dim_in, dim_out)
        ϕ[(i - 1) * dim_out + i] = 1
    end
    normalize!(ϕ)

    ψ = U * ϕ
    ρ = ψ * ψ'
    return ρ .* (dim_in / tr(ρ))          # Choi normalisation: tr(ρ) = dim_in
end

## A valid Choi operator

The result is positive semidefinite (a valid Choi matrix) with trace `dim_in`.

In [ ]:
choi = random_entangled_choi(2, 2; rng=Xoshiro(42))
(psd = eigmin(Hermitian(choi)) ≥ -1e-10, trace = real(tr(choi)))

## Detecting entanglement with the partial transpose

A pure bipartite state is entangled iff its partial transpose has a negative
eigenvalue (for pure states the PPT criterion is exact). The single example below
is NPT, and across many random draws every Choi matrix comes out entangled.

In [ ]:
pt = partial_transpose(choi, 2, [2, 2])
fraction_entangled = count(s -> !is_ppt(random_entangled_choi(2, 2; rng=Xoshiro(s)), 2, 2), 1:200) / 200
(min_eig_partial_transpose = round(eigmin(Hermitian(pt)), digits=6),
 entangled = !is_ppt(choi, 2, 2),
 fraction_entangled = fraction_entangled)